# Patient Journey Intelligence System (PJIS)
## Part 4: XGBoost Risk Prediction & SHAP Explainability
**Author:** Inderjeet  
**Tech Stack:** XGBoost, Scikit-Learn, SHAP TreeExplainer, ROC-AUC, PR-AUC


### 1. Model Training & Benchmark Comparison


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sys.path.append(os.path.abspath('../src'))
from predictive_models import RiskPredictionEngine
from explainability import ExplainableAIEngine

df = pd.read_csv('../data/processed_patient_master.csv')
rpe = RiskPredictionEngine(model_dir='../models')
X_train_scaled, X_test_scaled, y_train, y_test = rpe.prepare_data(df)
metrics = rpe.train_models()

# Performance table
pd.DataFrame([
    {
        'Model': name,
        'ROC-AUC': f"{m['ROC_AUC']:.4f}",
        'PR-AUC': f"{m['PR_AUC']:.4f}",
        'Accuracy': f"{m['Accuracy']:.2%}",
        'Recall': f"{m['Recall']:.2%}",
        'F1-Score': f"{m['F1_Score']:.4f}"
    } for name, m in metrics.items()
])


### 2. ROC Curves & Confusion Matrix


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
for name, m in metrics.items():
    axes[0].plot(m['FPR'], m['TPR'], label=f"{name} (AUC = {m['ROC_AUC']:.3f})")
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve Benchmark")
axes[0].legend()

# Confusion Matrix for XGBoost
cm = np.array(metrics['XGBoost']['Confusion_Matrix'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Pred Low', 'Pred High'],
            yticklabels=['Actual Low', 'Actual High'])
axes[1].set_title("XGBoost Confusion Matrix")
plt.tight_layout()
plt.show()


### 3. SHAP TreeExplainer & Global Attribution


In [ ]:
xai = ExplainableAIEngine(model_dir='../models')
xai.initialize_explainer(rpe.models['XGBoost'], X_test_scaled.iloc[:100])
global_importance = xai.get_global_feature_importance(top_n=15)

plt.figure(figsize=(9, 5))
sns.barplot(data=global_importance, x='Mean_Absolute_SHAP', y='Feature', palette='Blues_r')
plt.title("Top Global SHAP Clinical Feature Importances (XGBoost)")
plt.xlabel("Mean |SHAP Value|")
plt.show()


### 4. Local Patient Explanation (Waterfall)


In [ ]:
patient_sample = X_test_scaled.iloc[0]
explanation = xai.explain_patient(patient_sample)
print("Clinician Narrative:\n", explanation['Clinical_Narrative'])
print("\nTop Positive Risk Drivers:\n", explanation['Risk_Drivers'][['Feature', 'SHAP_Value']])
